# Re-etiquetado del Dataset `final_labelled_citing_sentences_all.txt`

Proceso:

1. Cargar el archivo `final_labelled_citing_sentences_all.txt`.
2. Asignar las columnas: `paper-id`, `published-date-in-ArXiv`, `paper-title`, `line-number`, `citing-sentence`, `label`.
3. Reemplazar las etiquetas originales de la columna `label` según el siguiente mapeo:

| Etiqueta original | Nueva etiqueta |
|---|---|
| `definition` | Background |
| `suggest` | Further Reading |
| `citing_paper_corroboration` | Evidence |
| `citing_paper_based_on` | Basis |
| `citing_paper_use` | Application |
| `citing_paper_extend` | Improvement / Modification |
| `cited_paper_propose` | Identification of the Originator |
| `cited_paper_weakness` | Gap |
| `compare` | Comparison |

4. Descartar cualquier fila cuya etiqueta original **no** esté en el mapeo.
5. Guardar el resultado en un nuevo archivo `.csv`.


## 1. Importar librerías

In [14]:
import pandas as pd
from pathlib import Path


## 2. Configuración de rutas y columnas

In [ ]:
# Nombre base del archivo.
BASE_NAME = "final_labelled_citing_sentences_all"

INPUT_PATH = Path(f"{BASE_NAME}.txt")
OUTPUT_PATH = Path(f"{BASE_NAME}.csv")

# Nombres de columnas ordenadas
COLUMN_NAMES = [
    "paper-id",
    "published-date-in-ArXiv",
    "paper-title",
    "line-number",
    "citing-sentence",
    "label",
]

# Mapeo de etiquetas originales -> nuevas etiquetas.
LABEL_MAPPING = {
    "definition": "Background",
    "suggest": "Further Reading",
    "citing_paper_corroboration": "Evidence",
    "citing_paper_based_on": "Basis",
    "citing_paper_use": "Application",
    "citing_paper_extend": "Improvement / Modification",
    "cited_paper_propose": "Identification of the Originator",
    "cited_paper_weakness": "Gap",
    "compare": "Comparison",
}


## 3. Carga del archivo

Las columnas están separadas por **tabulaciones** (se usa `sep="\t"`).

In [16]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo '{INPUT_PATH}'."
    )

df = pd.read_csv(
    INPUT_PATH,
    sep="\t",
    header=None,
    names=COLUMN_NAMES,
    engine="python",
    on_bad_lines="warn",
    encoding="utf-8",
    dtype={"paper-id": str, "line-number": str},
)

print(f"\nTotal de filas cargadas: {len(df)}")
df.head(10)



Total de filas cargadas: 1840820


,paper-id,published-date-in-ArXiv,paper-title,line-number,citing-sentence,label
0,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,10,Our results provide a unifying framework for ...,citing_paper_corroboration
1,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,11,"Indeed , in the lower range , canonical pebbl...",other
2,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,28,Map-graphs may be equivalently defined ( see ...,definition
3,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,41,The equivalence of maps-and-trees graphs and ...,other
4,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,42,In rigidity theory a foundational theorem of ...,judgement
5,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,43,Tay <citation> proved an analogous result for...,cited_paper_result
6,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,44,Rigidity by counts motivated interest in the ...,cited_paper_result
7,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,45,Tay <citation> used this condition to give a ...,cited_paper_propose
8,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,46,Haas <citation> studied decompositions in det...,cited_paper_result
9,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,47,A pebble game algorithm was first proposed in...,cited_paper_propose


## 4. Etiquetas originales

In [17]:
print("Valores únicos en la columna 'label' (antes del filtrado):")
print(df["label"].value_counts(dropna=False))


Valores únicos en la columna 'label' (antes del filtrado):
label
other                         511830
cited_paper_propose           243031
judgement                     215428
cited_paper_result            154394
citing_paper_use              115215
citing_paper_corroboration    113488
technical                      85374
trend                          66594
citing_paper_based_on          55878
definition                     55508
suggest                        51987
compare                        39364
cited_paper_success            34505
citing_paper_extend            28779
citing_paper_dominant          24823
contrast                       20909
cited_paper_weakness           15054
citing_paper_future             5439
cited_paper_dominant            3214
NaN                                6
Name: count, dtype: int64


## 5. Reemplazo de etiquetas y descarte de registros no mapeados

- Se limpian los espacios en blanco sobrantes en `label` antes de comparar.
- Se aplica `LABEL_MAPPING`.
- Toda fila cuya etiqueta original no exista en el diccionario se descarta.

In [18]:
# Limpiar posibles espacios en blanco alrededor del label
df["label"] = df["label"].astype(str).str.strip()

filas_antes = len(df)

# Nos quedamos solo con las filas cuya etiqueta original está en el mapeo
df_filtrado = df[df["label"].isin(LABEL_MAPPING.keys())].copy()

# Aplicamos el reemplazo
df_filtrado["label"] = df_filtrado["label"].map(LABEL_MAPPING)

filas_despues = len(df_filtrado)
filas_descartadas = filas_antes - filas_despues

print(f"Filas antes del filtrado:      {filas_antes}")
print(f"Filas después del filtrado:    {filas_despues}")
print(f"Filas descartadas:             {filas_descartadas}")


Filas antes del filtrado:      1840820
Filas después del filtrado:    718304
Filas descartadas:             1122516


## 6. Verificación del resultado

In [19]:
print("Distribución final de etiquetas (nuevas categorías):")
print(df_filtrado["label"].value_counts())

df_filtrado.head(10)


Distribución final de etiquetas (nuevas categorías):
label
Identification of the Originator    243031
Application                         115215
Evidence                            113488
Basis                                55878
Background                           55508
Further Reading                      51987
Comparison                           39364
Improvement / Modification           28779
Gap                                  15054
Name: count, dtype: int64


,paper-id,published-date-in-ArXiv,paper-title,line-number,citing-sentence,label
0,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,10,Our results provide a unifying framework for ...,Evidence
2,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,28,Map-graphs may be equivalently defined ( see ...,Background
7,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,45,Tay <citation> used this condition to give a ...,Identification of the Originator
9,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,47,A pebble game algorithm was first proposed in...,Identification of the Originator
10,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,48,"Berg and Jordan <citation> , provided the for...",Identification of the Originator
11,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,49,Lee and Streinu <citation> generalized the pe...,Identification of the Originator
12,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,52,Since the phrase `` with colors '' is necessa...,Further Reading
13,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,94,Our first result is a strengthening of the pe...,Improvement / Modification
14,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,109,These time and space bounds mean that our alg...,Improvement / Modification
16,0704.0002,2008-12-13,Sparsity-certifying Graph Decompositions,111,Since many of the relevant properties of the ...,Further Reading


## 7. Guardar el resultado como CSV

In [20]:
df_filtrado.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Archivo guardado en: {OUTPUT_PATH.resolve()}")
print(f"Total de registros guardados: {len(df_filtrado)}")


Archivo guardado en: C:\PERSONAL\Maestria IA\2026-02-01\Proyecto - Desarrollo de Soluciones\Microproyecto\microproyecto\data\final_labelled_citing_sentences_all.csv
Total de registros guardados: 718304
